# 01 — Data Exploration

Memverifikasi asumsi di Bagian 4 guideline sebelum menulis pipeline apa pun.

**Catatan:** angka coverage header di guideline (99%) ternyata terlalu tinggi karena 
dihitung dengan substring polos. Notebook ini memakai pola header yang benar dan 
menghasilkan angka yang lebih rendah — lihat bagian kelayakan chunking.

**Dataset:** Resume Dataset (livecareer.com scraping) — `data/raw/Resume.csv`

In [9]:
import pandas as pd

pd.set_option("display.max_colwidth", 200)

df = pd.read_csv("../data/raw/Resume.csv")

print("Shape :", df.shape)
print("Kolom :", df.columns.tolist())

Shape : (2484, 4)
Kolom : ['ID', 'Resume_str', 'Resume_html', 'Category']


### Apakah distribusi kategori cukup seimbang untuk dijadikan filter?

Metadata filtering hanya berguna kalau distribusinya tidak timpang ekstrem. Kategori dengan <20 dokumen tidak layak jadi filter tersendiri — hasil retrieval-nya akan terlalu sedikit untuk berarti.

**Ambang:** kategori terkecil harus ≥20 dokumen.

In [10]:
vc = df["Category"].value_counts()

print(vc)
print()
print(f"Jumlah kategori : {len(vc)}")
print(f"Terbesar        : {vc.max()} ({vc.idxmax()})")
print(f"Terkecil        : {vc.min()} ({vc.idxmin()})")
print(f"Rasio timpang   : {vc.max() / vc.min():.1f}x")

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
FINANCE                   118
ADVOCATE                  118
ACCOUNTANT                118
ENGINEERING               118
CHEF                      118
AVIATION                  117
FITNESS                   117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

Jumlah kategori : 24
Terbesar        : 120 (INFORMATION-TECHNOLOGY)
Terkecil        : 22 (BPO)
Rasio timpang   : 5.5x


### Temuan — distribusi kategori

**Ambang terpenuhi.** Kategori terkecil (BPO) punya 22 dokumen, di atas ambang 20. 
Semua 24 kategori layak dipakai sebagai filter metadata.

Distribusi relatif seimbang: 20 dari 24 kategori di rentang 96–120 dokumen. Rasio 
timpang 5,5x seluruhnya dari tiga outlier bawah — AGRICULTURE (63), AUTOMOBILE (36), 
BPO (22).

**Implikasi retrieval:** filter `category="BPO"` menyisakan ~130 chunk. Retrieve 
top-20 di ruang sekecil itu berarti mengambil ~15% kategori — reranking jadi 
penentu utama, bukan vector search.

**Implikasi evaluation:** golden set wajib memuat minimal satu query ke kategori 
kecil. Kalau semua query menyasar kategori besar, recall@5 terlihat bagus tapi 
tidak menguji kondisi sempit.

Tidak ada tindakan cleaning dari temuan ini — ketimpangan ini karakteristik data.

### Seberapa banyak cleaning yang dibutuhkan?

**Ambang baris sampah:** <500 karakter dianggap tidak layak — terlalu pendek untuk 
mengandung informasi resume yang bermakna.

In [11]:
lengths = df["Resume_str"].str.len()

print(lengths.describe())
print()
print("Baris < 500 karakter :", (lengths < 500).sum())
print("Duplikat Resume_str  :", df["Resume_str"].duplicated().sum())
print("Duplikat ID          :", df["ID"].duplicated().sum())
print("Null Resume_str      :", df["Resume_str"].isna().sum())

count     2484.000000
mean      6295.308776
std       2769.251458
min         21.000000
25%       5160.000000
50%       5886.500000
75%       7227.250000
max      38842.000000
Name: Resume_str, dtype: float64

Baris < 500 karakter : 1
Duplikat Resume_str  : 2
Duplikat ID          : 0
Null Resume_str      : 0


### Temuan — kualitas data

Data jauh lebih bersih dari perkiraan. Cleaning bisa minimal.

| Metrik | Nilai |
|---|---|
| Dokumen | 2.484 |
| Rata-rata / median | 6.295 / 5.886 karakter |
| Rentang | 21 – 38.842 karakter |
| Baris <500 karakter | 1 |
| Duplikat teks | 2 |
| Duplikat ID / null | 0 / 0 |

Mean (6.295) sedikit di atas median (5.886) — distribusi condong ke kanan, wajar 
untuk resume: sebagian kecil sangat panjang, mayoritas di kisaran serupa.

**Dua nilai ekstrem yang perlu ditangani:**

- **21 karakter** — satu baris, jelas sampah. Drop.
- **38.842 karakter** — 6x median. Bukan sampah, tapi akan menghasilkan chunk jauh 
  lebih banyak dari dokumen lain. Berpotensi mendominasi hasil retrieval kalau 
  tidak dibatasi.

**Keputusan untuk `loader.py`:**
1. Drop baris <500 karakter (1 baris)
2. Drop duplikat `Resume_str`, pertahankan yang pertama (2 baris)
3. Korpus final: **2.481 dokumen**
4. Catat dokumen >20.000 karakter untuk dipantau saat evaluasi retrieval

### Berapa biaya embedding korpus ini?

Menentukan apakah re-index berulang saat eksperimen chunking masuk akal secara biaya.

In [12]:
total_chars = df["Resume_str"].str.len().sum()
est_tokens = total_chars / 4
PRICE_PER_1M = 0.02   # text-embedding-3-small

print(f"Total karakter    : {total_chars:,}")
print(f"Estimasi token    : {est_tokens/1e6:.2f}M")
print(f"Biaya sekali index: ${est_tokens/1e6*PRICE_PER_1M:.4f}")
print(f"Biaya 10x index   : ${est_tokens/1e6*PRICE_PER_1M*10:.4f}")

Total karakter    : 15,637,547
Estimasi token    : 3.91M
Biaya sekali index: $0.0782
Biaya 10x index   : $0.7819


### Temuan biaya embedding

3,91M token ≈ **$0,0782** per satu kali index penuh dengan `text-embedding-3-small`.

Bahkan 10x re-index selama eksperimen hanya ~$0,78 — di bawah 8% saldo. **Biaya 
embedding bukan kendala**, jadi ablation chunking bisa dijalankan bebas.

Yang justru mahal adalah query LLM saat development. Prioritas penghematan ada di 
sana, bukan di sini.

Tetap terapkan cache ke `cache/embeddings.parquet` — bukan untuk menghemat uang, 
tapi menghemat **waktu tunggu** saat iterasi debugging.

### Apakah section aware chunking layak?

**Ambang:** >90% coverage untuk Experience, Education, Skills. Di bawah itu, 
strategi Bagian 4.3 guideline harus direvisi.

**Metode diperbaiki.** Perhitungan awal memakai `str.contains("Experience")` polos 
— ini menghitung kata yang muncul di mana pun, termasuk di tengah kalimat 
("5 years of experience"), sehingga hasilnya terlalu optimis.

Perhitungan di sini memakai pola `\s{2,}Header\s{2,}` **setelah normalisasi** 
`\xa0`, `\t`, `\u2028` → spasi. Ini mendeteksi header sebagai penanda struktural, 
bukan sekadar kemunculan kata.

In [13]:
import re

headers = [
    "Summary", "Highlights", "Accomplishments", "Experience",
    "Education", "Skills", "Professional Summary",
    "Core Qualifications", "Certifications",
]

# Normalisasi dulu: \xa0, \t, \u2028 → spasi biasa
norm = df["Resume_str"].fillna("").str.replace(
    r"[\xa0\t\u2028\u2029]", " ", regex=True
)

for h in headers:
    pattern = rf"\s{{2,}}{re.escape(h)}\s{{2,}}"
    n = norm.str.contains(pattern, case=False, regex=True).sum()
    print(f"{h:22s} {n:5d}  ({n/len(df)*100:5.1f}%)")

Summary                 1345  ( 54.1%)
Highlights               875  ( 35.2%)
Accomplishments          765  ( 30.8%)
Experience              1719  ( 69.2%)
Education               1922  ( 77.4%)
Skills                  2324  ( 93.6%)
Professional Summary     465  ( 18.7%)
Core Qualifications      211  (  8.5%)
Certifications           285  ( 11.5%)


### Temuan — kelayakan section-aware chunking

**Ambang TIDAK terpenuhi.** Hanya Skills yang lolos.

| Section | Coverage | Lolos >90%? |
|---|---|---|
| Skills | 93,6% | Ya |
| Education | 77,4% | Tidak |
| Experience | 69,2% | Tidak |
| Summary | 54,1% | Tidak |
| Highlights | 35,2% | Tidak |
| Accomplishments | 30,8% | Tidak |
| Professional Summary | 18,7% | Tidak |
| Certifications | 11,5% | Tidak |
| Core Qualifications | 8,5% | Tidak |

Angka ini jauh di bawah klaim guideline (99%). Selisihnya menunjukkan seberapa 
menyesatkan pengukuran substring polos — sebagian besar kemunculan kata 
"Experience" ternyata bagian dari kalimat, bukan header.

**Penyebab coverage rendah:** dataset ini scraping dari template resume yang 
berbeda-beda. Sebagian pakai "Summary", sebagian "Professional Summary", sebagian 
tidak pakai header sama sekali.

**Keputusan: hybrid, bukan salah satu.**

Section-aware chunking **tetap dipakai**, tapi bukan sebagai satu-satunya jalur:

1. Coba deteksi header dengan pola spasi
2. Kalau ≥2 header terdeteksi → split per section
3. Kalau <2 → fallback `RecursiveCharacterTextSplitter`
4. Simpan `chunking_method` di payload Qdrant

Poin 4 penting: dengan mencatat metode per chunk, evaluation harness bisa 
membandingkan recall dokumen yang ter-split section vs yang fallback. Itu mengubah 
keterbatasan data jadi **temuan yang terukur**.

**Estimasi:** ~70% dokumen punya cukup header untuk section-aware, ~30% masuk 
fallback.

**Konsekuensi untuk guideline Bagian 4.3:** klaim "header sangat terdeteksi" harus 
dikoreksi. Ablation chunking bukan lagi "section-aware vs naive" tapi "hybrid vs 
naive murni" — dan hasilnya kemungkinan lebih tipis dari perkiraan awal.

### Bagaimana format teks mentahnya?

Regex splitter dibangun dari observasi ini, bukan asumsi. `repr()` dipakai agar 
karakter tak terlihat tampil eksplisit.

In [14]:
sample = df["Resume_str"].iloc[0]

print("Panjang:", len(sample))
print()
print(repr(sample[:2000]))

Panjang: 5442

'         HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory c

In [15]:
# Cek karakter kontrol / tak biasa di seluruh korpus
from collections import Counter

sus = Counter()
for t in df["Resume_str"].dropna():
    sus.update(re.findall(r"[\u2028\u2029\r\t\xa0]", t))

print("Karakter tak biasa yang ditemukan:")
for ch, n in sus.items():
    print(f"  {repr(ch):10s} {n:,}")

Karakter tak biasa yang ditemukan:
  '\xa0'     8,877
  '\t'       7,257
  '\u2028'   9


### Observasi format

**Temuan kritis: header TIDAK berada di baris terpisah.**

Seluruh resume 5.442 karakter hanya mengandung 2 kejadian `\n`. Header dipisahkan 
dari teks sekitarnya oleh **runs of spaces**, bukan newline:

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager...
...time management, leadership and performance assessment.        Experience      HR Administrator/Marketing Associate

Ini membatalkan pendekatan split berbasis baris (`text.split("\n")`) yang lazim 
dipakai untuk dokumen terstruktur. Splitter harus berbasis **pola spasi berurutan**.

**Karakteristik header:**
- Title Case (`Summary`, `Highlights`, `Experience`) — bukan kapital semua
- Diapit minimal 2 spasi di kedua sisi, umumnya 4–8
- Urutan konsisten: Summary → Highlights → Accomplishments → Experience → Education

**Karakter tak biasa di seluruh korpus:**

| Karakter | Jumlah | Tindakan |
|---|---|---|
| `\xa0` (non-breaking space) | 8.877 | Normalisasi ke spasi biasa |
| `\t` (tab) | 7.257 | Normalisasi ke spasi biasa |
| `\u2028` (line separator) | 9 | Normalisasi — jumlahnya sepele |

`\xa0` dan `\t` **wajib** dinormalisasi lebih dulu. Keduanya muncul di posisi yang 
sama dengan spasi pemisah header, sehingga regex berbasis spasi literal akan 
melewatkan header yang dipisah oleh tab atau nbsp.

**Artefak lain:** judul dokumen muncul dua kali di awal 
(`HR ADMINISTRATOR/MARKETING ASSOCIATE` lalu `HR ADMINISTRATOR`), sisa dari 
struktur HTML sumber.

**Keputusan untuk `chunker.py`:**

1. Normalisasi `\xa0`, `\t`, `\u2028` → spasi tunggal, **sebelum** deteksi header
2. Split dengan pola `\s{2,}(Header)\s{2,}`, bukan berbasis newline
3. Wajib `\b` word boundary — mencegah false positive dari teks berjalan
4. Fallback ke `RecursiveCharacterTextSplitter` bila tidak ada header terdeteksi

### Seberapa besar variasi panjang antar kategori?

Relevan untuk chunking: kategori dengan resume sangat panjang akan menghasilkan lebih banyak chunk, memengaruhi distribusi hasil retrieval.

In [16]:
stats = df.groupby("Category")["Resume_str"].apply(
    lambda s: pd.Series({
        "n": len(s),
        "mean_chars": s.str.len().mean(),
        "max_chars": s.str.len().max(),
    })
).unstack().sort_values("mean_chars", ascending=False)

print(stats.round(0).to_string())

                            n  mean_chars  max_chars
Category                                            
BPO                      22.0      7318.0    16223.0
INFORMATION-TECHNOLOGY  120.0      7228.0    20561.0
HEALTHCARE              115.0      6996.0    21316.0
PUBLIC-RELATIONS        111.0      6913.0    35933.0
CONSULTANT              115.0      6844.0    15411.0
HR                      110.0      6761.0    24834.0
CONSTRUCTION            112.0      6618.0    30055.0
AGRICULTURE              63.0      6605.0    14612.0
ADVOCATE                118.0      6575.0    20975.0
ENGINEERING             118.0      6476.0    19632.0
AUTOMOBILE               36.0      6321.0    22032.0
FINANCE                 118.0      6316.0    20698.0
ACCOUNTANT              118.0      6294.0    24695.0
BANKING                 115.0      6191.0    15896.0
AVIATION                117.0      6189.0    14060.0
DIGITAL-MEDIA            96.0      6095.0    24500.0
BUSINESS-DEVELOPMENT    120.0      6044.0    1

## Kesimpulan

| Asumsi guideline | Terverifikasi? | Angka aktual |
|---|---|---|
| 2.484 dokumen | Ya | 2.484 |
| Rata-rata ~6.295 karakter | Ya | 6.295 |
| 1 baris <500 karakter | Ya | 1 |
| 2 duplikat teks | Ya | 2 |
| Coverage Experience >90% | **Tidak** | 69,2% |
| Coverage Education >90% | **Tidak** | 77,4% |
| Coverage Skills >90% | Ya | 93,6% |
| Biaya embedding ~$0,08 | Ya | $0,0782 |

**Empat dari lima asumsi terverifikasi. Yang gagal adalah yang paling menentukan 
arsitektur.**

### Keputusan untuk `loader.py`
1. Normalisasi `\xa0`, `\t`, `\u2028` → spasi tunggal
2. Kolapskan spasi berulang **hanya setelah** deteksi header — spasi berulang 
   adalah sinyal struktural, bukan noise
3. Drop 1 baris <500 karakter, 2 duplikat teks → korpus final 2.481
4. Pakai `Resume_str`, bukan `Resume_html`

### Keputusan untuk `chunker.py`
1. Deteksi header dengan `\s{2,}(Header)\s{2,}`, **bukan** berbasis newline — 
   dokumen 5.442 karakter hanya punya 2 newline
2. Hybrid: section-aware bila ≥2 header, fallback bila kurang
3. Simpan `chunking_method` di payload untuk analisis komparatif
4. Word boundary wajib, mencegah false positive

### Yang harus dikoreksi di guideline
- Bagian 4.3: coverage 99% → angka aktual di tabel atas
- Bagian 6.3 tabel A: ablation jadi "hybrid vs naive murni"
- Bagian 4.4: tambah field `chunking_method` ke payload schema